# Initialize Database

In [2]:
from pathlib import Path

def find_project_root() -> Path:
    path = Path.cwd()
    while path != path.parent:
        if (path / "pyproject.toml").exists():
            return path
        path = path.parent
    raise RuntimeError("Could not find project root")

database = str(find_project_root() / "data" / "autogc.db")

## EQ setup

In [3]:
import logging

logging.basicConfig(level = logging.INFO)
logger = logging.getLogger(__name__)

## Station Setup

In [4]:

from autogc_validation.database.models import *
from autogc_validation.database.operations import insert, get_table, get_active_canister_concentrations, delete
from autogc_validation.database.enums import *
site_eq = Site(site_id=490353015, name_short = 'EQ', name_long = "Utah Technical Center", lat = 40.7770994964404, long = -111.9450044213331, date_started="2023-10-01 00:00:00")
insert(database, site_eq)

ERROR:autogc_validation.database.conn.connection:Exception during transaction
Traceback (most recent call last):
  File "D:\autogc_validation\src\autogc_validation\database\conn\connection.py", line 37, in transaction
    yield conn
  File "D:\autogc_validation\src\autogc_validation\database\operations\insert.py", line 56, in insert
    conn.execute(sql, values)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
sqlite3.IntegrityError: UNIQUE constraint failed: sites.site_id


False

## CVS setup

#### Primary Canister Setup

##### Primary CC524930-0626

In [ ]:
cvs1 = PrimaryCanister(primary_canister_id= "CC524930-0626", canister_type="CVS", expiration_date="2026-06-01 00:00:00")
insert(database, cvs1)
cvs1_concentrations = {
    # Channel A
    "ETHANE": 0.525,
    "PROPANE": 0.34,
    "N-BUTANE": 0.253,
    "ACETYLENE": 0.525,
    "N-PENTANE": 0.204,
    "1,3-BUTADIENE": 0.263,
    "2-METHYLPENTANE": 0.17,
    "1-HEXENE": 0.17,

    # Channel B
    "N-HEXANE": 0.167,
    "BENZENE": 0.175,
    "TOLUENE": 0.146,
    "M&P-XYLENE": 0.131,
    "N-PROPYLBENZENE": 0.116,
    "1,2,4-TRI-M-BENZENE": 0.113,
    "P-DIETHYLBENZENE": 0.102,
}
for compound, concentration in cvs1_concentrations.items():
    aqs = name_to_aqs(compound.capitalize())
    p = CanisterConcentration(primary_canister_id = cvs1.primary_canister_id, aqs_code = aqs, concentration=concentration, units = "ppmv", canister_type="CVS")
    insert(database, p)
cvs1_concentrations = get_table(database, CanisterConcentration.__tablename__)
cvs1_concentrations

##### Site Canister CC524930-0626

In [ ]:
eq_cvs1 = SiteCanister(site_canister_id="3667", site_id=site_eq.site_id, primary_canister_id=cvs1.primary_canister_id, dilution_ratio=0.00189,date_on="2024-09-07 00:00:00", date_off="2025-01-22 09:10:00")
insert(database, eq_cvs1)
eq_cvs2 = SiteCanister(site_canister_id="3847", site_id=site_eq.site_id, primary_canister_id=cvs1.primary_canister_id, dilution_ratio=0.00188,date_on="2025-01-22 09:15:00", date_off="2025-04-21 08:29:59")
insert(database, eq_cvs2)
eq_cvs3 = SiteCanister(site_canister_id="3849", site_id=site_eq.site_id, primary_canister_id=cvs1.primary_canister_id, dilution_ratio=0.00188,date_on="2025-04-21 08:30:00", date_off="2025-08-15 08:20:59")
insert(database, eq_cvs3)

cvs_conc = get_active_canister_concentrations(database, site_eq.site_id, "CVS", date = "2025-04-01", output_unit = "ppbc")
eq_df = get_table(database, SiteCanister.__tablename__)
eq_df

## LCS Setup

#### LCS CC177206-0125

In [9]:
lcs1 = PrimaryCanister(primary_canister_id= "CC177206-0125", canister_type="LCS", expiration_date="2025-01-01 00:00:00")
insert(database, lcs1)
lcs1_dr = .005
lcs1_concentrations = {
    # Channel A
    "PROPANE": 0.369,
    "N-BUTANE": 0.279,
    "ETHANE": 0.51,
    "ACETYLENE": 0.557,
    "N-PENTANE": 0.228,
    "1,3-BUTADIENE": 0.28,
    "2-METHYLPENTANE": 0.176,
    "1-HEXENE": 0.18,

    # Channel B
    "N-HEXANE": 0.184,
    "BENZENE": 0.18,
    "TOLUENE": 0.142,
    "M&P-XYLENE": 0.116,
    "N-PROPYLBENZENE": 0.106,
    "1,2,4-TRI-M-BENZENE": 0.103,
    "P-DIETHYLBENZENE": 0.084,
}
for compound, concentration in lcs1_concentrations.items():
    aqs = name_to_aqs(compound.capitalize())
    p = CanisterConcentration(primary_canister_id = lcs1.primary_canister_id, aqs_code = aqs, concentration=concentration, units = "ppmv", canister_type="LCS")
    insert(database, p)
lcs1_concentrations = get_table(database, CanisterConcentration.__tablename__)
lcs1_concentrations

ERROR:autogc_validation.database.conn.connection:Exception during transaction
Traceback (most recent call last):
  File "D:\autogc_validation\src\autogc_validation\database\conn\connection.py", line 37, in transaction
    yield conn
  File "D:\autogc_validation\src\autogc_validation\database\operations\insert.py", line 56, in insert
    conn.execute(sql, values)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
sqlite3.IntegrityError: UNIQUE constraint failed: primary_canisters.primary_canister_id
ERROR:autogc_validation.database.conn.connection:Exception during transaction
Traceback (most recent call last):
  File "D:\autogc_validation\src\autogc_validation\database\conn\connection.py", line 37, in transaction
    yield conn
  File "D:\autogc_validation\src\autogc_validation\database\operations\insert.py", line 56, in insert
    conn.execute(sql, values)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
sqlite3.IntegrityError: UNIQUE constraint failed: primary_canister_concentration.primary_canister_id, primary_canister_con

,primary_canister_id,aqs_code,concentration,units,canister_type
0,CC524930-0626,43202,0.5250,ppmv,CVS
1,CC524930-0626,43204,0.3400,ppmv,CVS
2,CC524930-0626,43212,0.2530,ppmv,CVS
3,CC524930-0626,43206,0.5250,ppmv,CVS
4,CC524930-0626,43220,0.2040,ppmv,CVS
...,...,...,...,...,...
97,CC177206-1226,45202,0.1526,ppmv,LCS
98,CC177206-1226,45109,0.1309,ppmv,LCS
99,CC177206-1226,45209,0.1128,ppmv,LCS
100,CC177206-1226,45208,0.1042,ppmv,LCS


##### Site Cans LCS CC177206-0125 

In [10]:
eq_lcs1 = SiteCanister(site_canister_id="3733", site_id=site_eq.site_id, primary_canister_id=lcs1.primary_canister_id, dilution_ratio=lcs1_dr,date_on="2024-11-06 10:00:00", date_off="2025-06-03 12:10:00")
insert(database, eq_lcs1)


ERROR:autogc_validation.database.conn.connection:Exception during transaction
Traceback (most recent call last):
  File "D:\autogc_validation\src\autogc_validation\database\conn\connection.py", line 37, in transaction
    yield conn
  File "D:\autogc_validation\src\autogc_validation\database\operations\insert.py", line 56, in insert
    conn.execute(sql, values)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
sqlite3.IntegrityError: UNIQUE constraint failed: site_canisters.site_canister_id


False

In [11]:
eq_df = get_table(database, SiteCanister.__tablename__)
eq_df

,site_canister_id,site_id,primary_canister_id,dilution_ratio,date_on,date_off
0,3667,490353015,CC524930-0626,0.00189,2024-09-07 00:00:00,2025-01-22 09:10:00
1,3847,490353015,CC524930-0626,0.00188,2025-01-22 09:15:00,2025-04-21 08:29:59
2,3849,490353015,CC524930-0626,0.00188,2025-04-21 08:30:00,2025-08-15 08:20:59
3,3733,490353015,CC177206-0125,0.00500,2024-11-06 10:00:00,2025-06-03 12:10:00
4,00000,490353015,CC731198-0126,0.00130,2024-03-08 00:00:00,2025-10-31 08:54:59
5,49054,490353015,CC731198-0126,0.00130,2025-10-31 08:55:00,NaN
6,2444,490353015,CC177206-1226,0.00500,2025-06-03 12:25:00,2025-11-10 11:59:59


#### LCS CC177206-1226

In [12]:
lcs2 = PrimaryCanister(primary_canister_id= "CC177206-1226", canister_type="LCS", expiration_date="2026-12-01 00:00:00")
insert(database, lcs2)
lcs2_dr = .005
lcs2_concentrations = {
    "ETHANE":             0.4955,
    "PROPANE":            0.36,
    "N-BUTANE":           0.268,
    "ACETYLENE":          0.5868,
    "N-PENTANE":          0.224,
    "1,3-BUTADIENE":      0.2708,
    "2-METHYLPENTANE":    0.1745,
    "1-HEXENE":           0.1735,
    "N-HEXANE":           0.1755,
    "BENZENE":            0.1802,
    "TOLUENE":            0.1526,
    "M&P-XYLENE":         0.1309,
    "N-PROPYLBENZENE":    0.1128,
    "1,2,4-TRI-M-BENZENE": 0.1042,
    "P-DIETHYLBENZENE":   0.0909,
}
for compound, concentration in lcs2_concentrations.items():
    aqs = name_to_aqs(compound.capitalize())
    p = CanisterConcentration(primary_canister_id = lcs2.primary_canister_id, aqs_code = aqs, concentration=concentration, units = "ppmv", canister_type="LCS")
    insert(database, p)
lcs2_concentrations = get_table(database, CanisterConcentration.__tablename__)
lcs2_concentrations

ERROR:autogc_validation.database.conn.connection:Exception during transaction
Traceback (most recent call last):
  File "D:\autogc_validation\src\autogc_validation\database\conn\connection.py", line 37, in transaction
    yield conn
  File "D:\autogc_validation\src\autogc_validation\database\operations\insert.py", line 56, in insert
    conn.execute(sql, values)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
sqlite3.IntegrityError: UNIQUE constraint failed: primary_canisters.primary_canister_id
ERROR:autogc_validation.database.conn.connection:Exception during transaction
Traceback (most recent call last):
  File "D:\autogc_validation\src\autogc_validation\database\conn\connection.py", line 37, in transaction
    yield conn
  File "D:\autogc_validation\src\autogc_validation\database\operations\insert.py", line 56, in insert
    conn.execute(sql, values)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
sqlite3.IntegrityError: UNIQUE constraint failed: primary_canister_concentration.primary_canister_id, primary_canister_con

,primary_canister_id,aqs_code,concentration,units,canister_type
0,CC524930-0626,43202,0.5250,ppmv,CVS
1,CC524930-0626,43204,0.3400,ppmv,CVS
2,CC524930-0626,43212,0.2530,ppmv,CVS
3,CC524930-0626,43206,0.5250,ppmv,CVS
4,CC524930-0626,43220,0.2040,ppmv,CVS
...,...,...,...,...,...
97,CC177206-1226,45202,0.1526,ppmv,LCS
98,CC177206-1226,45109,0.1309,ppmv,LCS
99,CC177206-1226,45209,0.1128,ppmv,LCS
100,CC177206-1226,45208,0.1042,ppmv,LCS


In [31]:
# Comparing concentrations between LCS CC177206-0125 and LCS CC177206-1226
import pandas as pd
lcs0125 = {
    # Channel A
    "PROPANE": 0.369,
    "N-BUTANE": 0.279,
    "ETHANE": 0.51,
    "ACETYLENE": 0.557,
    "N-PENTANE": 0.228,
    "1,3-BUTADIENE": 0.28,
    "2-METHYLPENTANE": 0.176,
    "1-HEXENE": 0.18,

    # Channel B
    "N-HEXANE": 0.184,
    "BENZENE": 0.18,
    "TOLUENE": 0.142,
    "M&P-XYLENE": 0.116,
    "N-PROPYLBENZENE": 0.106,
    "1,2,4-TRI-M-BENZENE": 0.103,
    "P-DIETHYLBENZENE": 0.084,
}
lcs1226 = {
    "ETHANE":             0.4955,
    "PROPANE":            0.36,
    "N-BUTANE":           0.268,
    "ACETYLENE":          0.5868,
    "N-PENTANE":          0.224,
    "1,3-BUTADIENE":      0.2708,
    "2-METHYLPENTANE":    0.1745,
    "1-HEXENE":           0.1735,
    "N-HEXANE":           0.1755,
    "BENZENE":            0.1802,
    "TOLUENE":            0.1526,
    "M&P-XYLENE":         0.1309,
    "N-PROPYLBENZENE":    0.1128,
    "1,2,4-TRI-M-BENZENE": 0.1042,
    "P-DIETHYLBENZENE":   0.0909,
}

lcs_df = pd.DataFrame({
    'lcs0125': pd.Series(lcs0125),
    'lcs1226': pd.Series(lcs1226)
})
lcs_df["percent_diff"] = (lcs_df['lcs0125'] - lcs_df['lcs1226'])/lcs_df['lcs0125']*100
display(lcs_df)


,lcs0125,lcs1226,percent_diff
"1,2,4-TRI-M-BENZENE",0.103,0.1042,-1.165049
"1,3-BUTADIENE",0.280,0.2708,3.285714
1-HEXENE,0.180,0.1735,3.611111
2-METHYLPENTANE,0.176,0.1745,0.852273
ACETYLENE,0.557,0.5868,-5.350090
BENZENE,0.180,0.1802,-0.111111
ETHANE,0.510,0.4955,2.843137
M&P-XYLENE,0.116,0.1309,-12.844828
N-BUTANE,0.279,0.2680,3.942652
N-HEXANE,0.184,0.1755,4.619565


##### Site Cans LCS CC177206-1226 

In [14]:
eq_lcs2 = SiteCanister(site_canister_id="2444", site_id=site_eq.site_id, primary_canister_id=lcs2.primary_canister_id, dilution_ratio=lcs2_dr,date_on="2025-06-03 12:25:00", date_off="2025-11-10 11:59:59")
insert(database, eq_lcs2)

ERROR:autogc_validation.database.conn.connection:Exception during transaction
Traceback (most recent call last):
  File "D:\autogc_validation\src\autogc_validation\database\conn\connection.py", line 37, in transaction
    yield conn
  File "D:\autogc_validation\src\autogc_validation\database\operations\insert.py", line 56, in insert
    conn.execute(sql, values)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
sqlite3.IntegrityError: UNIQUE constraint failed: site_canisters.site_canister_id


False

In [15]:
eq_df = get_table(database, SiteCanister.__tablename__)
eq_df

,site_canister_id,site_id,primary_canister_id,dilution_ratio,date_on,date_off
0,3667,490353015,CC524930-0626,0.00189,2024-09-07 00:00:00,2025-01-22 09:10:00
1,3847,490353015,CC524930-0626,0.00188,2025-01-22 09:15:00,2025-04-21 08:29:59
2,3849,490353015,CC524930-0626,0.00188,2025-04-21 08:30:00,2025-08-15 08:20:59
3,3733,490353015,CC177206-0125,0.00500,2024-11-06 10:00:00,2025-06-03 12:10:00
4,00000,490353015,CC731198-0126,0.00130,2024-03-08 00:00:00,2025-10-31 08:54:59
5,49054,490353015,CC731198-0126,0.00130,2025-10-31 08:55:00,NaN
6,2444,490353015,CC177206-1226,0.00500,2025-06-03 12:25:00,2025-11-10 11:59:59


## RTS

#### Primary CC731198-0126

In [ ]:
rts1 = PrimaryCanister(primary_canister_id= "CC731198-0126", canister_type="RTS", expiration_date="2026-01-01 00:00:00")
insert(database, rts1)
rts1_dr = .0013
rts1_concentrations = {
    # Channel A
    "ETHANE": 1070,
    "ETHYLENE": 1070,
    "PROPANE": 1040,
    "PROPYLENE": 1010,
    "ISO-BUTANE": 1060,
    "N-BUTANE": 1060,
    "ACETYLENE": 520,
    "TRANS-2-BUTENE": 1060,
    "1-BUTENE": 1060,
    "CIS-2-BUTENE": 1050,
    "CYCLOPENTANE": 1080,
    "ISO-PENTANE": 1100,
    "N-PENTANE": 1090,
    "1,3-BUTADIENE": 990,
    "TRANS-2-PENTENE": 1090,
    "1-PENTENE": 1130,
    "CIS-2-PENTENE": 1150,
    "2,2-DIMETHYLBUTANE": 1100,
    "2,3-DIMETHYLBUTANE": 1070,
    "2-METHYLPENTANE": 1080,
    "3-METHYLPENTANE": 1070,
    "ISOPRENE": 800,
    "1-HEXENE": 1060,

    # Channel B
    "N-HEXANE": 1040,
    "METHYLCYCLOPENTANE": 1070,
    "2,4-DIMETHYLPENTANE": 1070,
    "BENZENE": 1070,
    "CYCLOHEXANE": 1060,
    "2-METHYLHEXANE": 1030,
    "2,3-DIMETHYLPENTANE": 1080,
    "3-METHYLHEXANE": 1070,
    "2,2,4-TRIMETHYLPENTANE": 1060,
    "N-HEPTANE": 1100,
    "METHYLCYCLOHEXANE": 1060,
    "2,3,4-TRIMETHYLPENTANE": 1080,
    "TOLUENE": 1080,
    "2-METHYLHEPTANE": 1080,
    "3-METHYLHEPTANE": 1080,
    "N-OCTANE": 1200,
    "ETHYLBENZENE": 1070,
    "M&P-XYLENE": 1080,
    "STYRENE": 970,
    "O-XYLENE": 1070,
    "N-NONANE": 1090,
    "ISO-PROPYLBENZENE": 1070,
    "N-PROPYLBENZENE": 1040,
    "M-ETHYLTOLUENE": 1080,
    "P-ETHYLTOLUENE": 1010,
    "1,3,5-TRI-M-BENZENE": 1070,
    "O-ETHYLTOLUENE": 1050,
    "1,2,4-TRI-M-BENZENE": 1040,
    "N-DECANE": 1070,
    "1,2,3-TRI-M-BENZENE": 1020,
    "M-DIETHYLBENZENE": 1020,
    "P-DIETHYLBENZENE": 1030,
    "N-UNDECANE": 1030,
    "N-DODECANE": 980,
}
for compound, concentration in rts1_concentrations.items():
    aqs = name_to_aqs(compound.capitalize())
    p = CanisterConcentration(primary_canister_id = rts1.primary_canister_id, aqs_code = aqs, concentration=concentration, units = "ppbc", canister_type="RTS")
    insert(database, p)
rts1_concentrations = get_table(database, CanisterConcentration.__tablename__)
rts1_concentrations

In [ ]:
eq_rts1 = SiteCanister(site_canister_id="00000", site_id=site_eq.site_id, primary_canister_id=rts1.primary_canister_id, dilution_ratio=0.0013,date_on="2024-03-08 00:00:00", date_off= "2025-10-31 08:54:59")
eq_rts2 = SiteCanister(site_canister_id="49054", site_id=site_eq.site_id, primary_canister_id=rts1.primary_canister_id, dilution_ratio=0.0013,date_on="2025-10-31 08:55:00", date_off= None)
insert(database, eq_rts1)
insert(database, eq_rts2)
eq_df = get_table(database, SiteCanister.__tablename__)
eq_df

## MDL

## March 2025

In [ ]:
mdl_dict = {
    # Channel A
    "ETHANE": 0.1751,
    "ETHYLENE": 0.1108,
    "PROPANE": 0.1818,
    "PROPYLENE": 0.0731,
    "ISO-BUTANE": 0.1571,
    "N-BUTANE": 0.1362,
    "ACETYLENE": 0.0682,
    "TRANS-2-BUTENE": 0.0433,
    "1-BUTENE": 0.0592,
    "CIS-2-BUTENE": 0.0396,
    "CYCLOPENTANE": 0.0339,
    "ISO-PENTANE": 0.0614,
    "N-PENTANE": 0.0292,
    "1,3-BUTADIENE": 0.0544,
    "TRANS-2-PENTENE": 0.0524,
    "1-PENTENE": 0.0454,
    "CIS-2-PENTENE": 0.0349,
    "2,2-DIMETHYLBUTANE": 0.0393,
    "2,3-DIMETHYLBUTANE": 0.043,
    "2-METHYLPENTANE": 0.0427,
    "3-METHYLPENTANE": 0.0371,
    "ISOPRENE": 0.0261,
    "1-HEXENE": 0.0238,

    # Channel B
    "N-HEXANE": 0.1719,
    "METHYLCYCLOPENTANE": 0.0852,
    "2,4-DIMETHYLPENTANE": 0.0664,
    "BENZENE": 0.0531,
    "CYCLOHEXANE": 0.0844,
    "2-METHYLHEXANE": 0.0785,
    "2,3-DIMETHYLPENTANE": 0.1252,
    "3-METHYLHEXANE": 0.1086,
    "2,2,4-TRIMETHYLPENTANE": 0.1032,
    "N-HEPTANE": 0.0733,
    "METHYLCYCLOHEXANE": 0.1688,
    "2,3,4-TRIMETHYLPENTANE": 0.0923,
    "TOLUENE": 0.1074,
    "2-METHYLHEPTANE": 0.0835,
    "3-METHYLHEPTANE": 0.0806,
    "N-OCTANE": 0.0584,
    "ETHYLBENZENE": 0.0647,
    "M&P-XYLENE": 0.1121,
    "STYRENE": 0.0642,
    "O-XYLENE": 0.0895,
    "N-NONANE": 0.0639,
    "ISO-PROPYLBENZENE": 0.0546,
    "ALPHA-PINENE": 0.0901,
    "N-PROPYLBENZENE": 0.0398,
    "M-ETHYLTOLUENE": 0.0541,
    "P-ETHYLTOLUENE": 0.0756,
    "1,3,5-TRI-M-BENZENE": 0.0949,
    "O-ETHYLTOLUENE": 0.1373,
    "BETA-PINENE": 0.121,
    "1,2,4-TRI-M-BENZENE": 0.1054,
    "N-DECANE": 0.1122,
    "1,2,3-TRI-M-BENZENE": 0.1355,
    "M-DIETHYLBENZENE": 0.0882,
    "P-DIETHYLBENZENE": 0.0881,
    "N-UNDECANE": 0.0465,
    "N-DODECANE": 0.0972,
}
for compound, mdl in mdl_dict.items():
    aqs = name_to_aqs(compound.capitalize())
    m = MDL(site_id = site_eq.site_id, aqs_code = aqs, concentration = mdl, units = "ppbc", date_on = "2024-09-03 00:00:00", date_off = "2025-05-09 23:59:59")
    insert(database, m)
mdl_eq = get_table(database, MDL.__tablename__)
mdl_eq

## May 2025


In [ ]:
mdls_20250510 = {
    "ETHANE": 0.0814,
    "ETHYLENE": 0.0645,
    "PROPANE": 0.0843,
    "PROPYLENE": 0.0702,
    "ISO-BUTANE": 0.0882,
    "N-BUTANE": 0.0575,
    "ACETYLENE": 0.0437,
    "TRANS-2-BUTENE": 0.0334,
    "1-BUTENE": 0.0264,
    "CIS-2-BUTENE": 0.0256,
    "CYCLOPENTANE": 0.0272,
    "ISO-PENTANE": 0.0259,
    "N-PENTANE": 0.0295,
    "1,3-BUTADIENE": 0.0354,
    "TRANS-2-PENTENE": 0.0303,
    "1-PENTENE": 0.0478,
    "CIS-2-PENTENE": 0.0380,
    "2,2-DIMETHYLBUTANE": 0.0389,
    "2,3-DIMETHYLBUTANE": 0.0311,
    "2-METHYLPENTANE": 0.0317,
    "3-METHYLPENTANE": 0.0362,
    "ISOPRENE": 0.0278,
    "2-METHYL-1-PENTENE": 0.0437,
    "1-HEXENE": 0.0294,
    "N-HEXANE": 0.0796,
    "METHYLCYCLOPENTANE": 0.0422,
    "2,4-DIMETHYLPENTANE": 0.0566,
    "BENZENE": 0.0386,
    "CYCLOHEXANE": 0.1993,
    "2-METHYLHEXANE": 0.0582,
    "2,3-DIMETHYLPENTANE": 0.1279,
    "3-METHYLHEXANE": 0.1325,
    "2,2,4-TRIMETHYLPENTANE": 0.0769,
    "N-HEPTANE": 0.1030,
    "METHYLCYCLOHEXANE": 0.0426,
    "2,3,4-TRIMETHYLPENTANE": 0.0390,
    "TOLUENE": 0.1299,
    "2-METHYLHEPTANE": 0.0733,
    "3-METHYLHEPTANE": 0.0700,
    "N-OCTANE": 0.0758,
    "ETHYLBENZENE": 0.0430,
    "M&P-XYLENE": 0.0737,
    "STYRENE": 0.0839,
    "O-XYLENE": 0.0766,
    "N-NONANE": 0.1047,
    "ISO-PROPYLBENZENE": 0.0342,
    "ALPHA-PINENE": 0.2166,
    "N-PROPYLBENZENE": 0.0611,
    "M-ETHYLTOLUENE": 0.0663,
    "P-ETHYLTOLUENE": 0.0619,
    "1,3,5-TRI-M-BENZENE": 0.0632,
    "O-ETHYLTOLUENE": 0.0576,
    "BETA-PINENE": 0.0934,
    "1,2,4-TRI-M-BENZENE": 0.0858,
    "N-DECANE": 0.1940,
    "1,2,3-TRI-M-BENZENE": 0.0382,
    "M-DIETHYLBENZENE": 0.1000,
    "P-DIETHYLBENZENE": 0.0682,
    "N-UNDECANE": 0.0486,
    "N-DODECANE": 0.0667
}
for compound, mdl in mdls_20250510.items():
    aqs = name_to_aqs(compound.capitalize())
    m = MDL(site_id = site_eq.site_id, aqs_code = aqs, concentration = mdl, units = "ppbc", date_on = "2025-05-10 00:00:00", date_off = "2025-09-06 23:59:59")
    insert(database, m)
mdl_eq = get_table(database, MDL.__tablename__)
mdl_eq

## Backup database

Run this cell after making any changes to dump the database to `data/autogc.sql`.
Then commit the file to save a restorable snapshot.

In [ ]:
from autogc_validation.database.management import dump_database

dump_database(
    database_path=database,
    output_path=find_project_root() / "data" / "autogc.sql",
)
print("Done. Remember to commit data/autogc.sql.")